In [ ]:
# In[1]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Load CSVs
metric_df = pd.read_csv("metric.csv")
trace_df = pd.read_csv("trace.csv")
log_df = pd.read_csv("log.csv")

# Convert timestamp to timezone-aware datetimes (UTC epoch -> Asia/Shanghai)
for df in (metric_df, trace_df, log_df):
    if 'timestamp' in df.columns:
        df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Standardize KPI column name and ensure numeric 'value'
metric_df = metric_df.rename(columns={'kpi_name': 'KPI_name'})
trace_df = trace_df.rename(columns={'trace_name': 'KPI_name'})
log_df = log_df.rename(columns={'log_name': 'KPI_name'})

for df in (metric_df, trace_df, log_df):
    df['value'] = pd.to_numeric(df['value'], errors='coerce')

# Aggregation function: count and global P95 per (cmdb_id, KPI_name)
metric_agg = (
    metric_df.groupby(['cmdb_id', 'KPI_name'])['value']
    .agg(count='count', p95_value=lambda x: x.quantile(0.95))
    .reset_index()
)
metric_agg['source'] = 'metric'

trace_agg = (
    trace_df.groupby(['cmdb_id', 'KPI_name'])['value']
    .agg(count='count', p95_value=lambda x: x.quantile(0.95))
    .reset_index()
)
trace_agg['source'] = 'trace'

log_agg = (
    log_df.groupby(['cmdb_id', 'KPI_name'])['value']
    .agg(count='count', p95_value=lambda x: x.quantile(0.95))
    .reset_index()
)
log_agg['source'] = 'log'

# Combine and select required columns
combined = pd.concat([metric_agg, trace_agg, log_agg], ignore_index=True)
combined = combined[['source', 'cmdb_id', 'KPI_name', 'count', 'p95_value']]

# Top 20 by count descending
combined_top20 = combined.sort_values(by='count', ascending=False).head(20).reset_index(drop=True)

# Full list of unique cmdb_id values across all three files
unique_cmdbs = pd.Series(
    sorted(
        set(metric_df['cmdb_id'].dropna().unique()).union(
            set(trace_df['cmdb_id'].dropna().unique()),
            set(log_df['cmdb_id'].dropna().unique())
        )
    ),
    name='cmdb_id'
).to_frame()

# Display compact outputs: top 20 summary and unique cmdb ids
combined_top20, unique_cmdbs

```
Out[1]:
```
```python
# Summarize the previously computed results in plain English.
# Reuse existing variables: combined_top20, unique_cmdbs

num_unique_cmdbs = int(unique_cmdbs.shape[0])
top_counts = combined_top20['count'].unique()
source_breakdown = combined_top20['source'].value_counts().to_dict()
example_cmdbs = ', '.join(unique_cmdbs['cmdb_id'].head(10).tolist())

summary = (
    f"Summary:\n"
    f"- Top 20 grouped rows (by count) were computed across metric/trace/log. "
    f"All rows in the top-20 have count = {int(top_counts[0])}.\n"
    f"- Source breakdown within top-20: {source_breakdown} (e.g., many 'trace' entries; also 'metric' for IG01 JVM memory KPIs and 'log' for apache02).\n"
    f"- Total unique cmdb_id values across all three files: {num_unique_cmdbs}.\n"
    f"- Example cmdb_ids (first 10): {example_cmdbs}."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(    source   cmdb_id                                   KPI_name  count     p95_value
0    trace  dockerB1                   trace.self.duration_mean     31  6.026800e-02
1    trace  dockerA2                       trace.self.row_count     31  1.417500e+03
2    trace  dockerB1                  trace.from_MG02.row_count     31  1.865000e+02
3      log  apache02                              log.row_count     31  5.010000e+02
4   metric      IG01   JVM-Memory_7778_JVM_Memory_HeapMemoryMax     31  2.040136e+09
5      log  apache02                            log.error_count     31  0.000000e+00
6   metric      IG01  JVM-Memory_7778_JVM_Memory_HeapMemoryUsed     31  8.476101e+08
7    trace  dockerB2                       trace.self.row_count     31  1.077500e+03
8    trace  dockerA1               trace.from_MG01.duration_p95     31  1.052200e+00
9    trace  dockerA2              trace.from_MG02.duration_mean     31  2.437500e-01
10   trace  dockerA1              trace.from_MG02.duration_mean     31  1.805760e-01
11   trace  dockerA1               trace.from_MG02.duration_p95     31  1.028025e+00
12   trace  dockerA2               trace.from_MG02.duration_p95     31  1.025475e+00
13   trace  dockerA2                  trace.from_MG02.row_count     31  9.750000e+01
14   trace  dockerA1                   trace.self.duration_mean     31  2.146300e-02
15   trace  dockerA1                    trace.self.duration_p95     31  8.000000e-03
16   trace  dockerA1                       trace.self.row_count     31  1.356000e+03
17   trace  dockerA2              trace.from_MG01.duration_mean     31  1.495405e-01
18   trace  dockerA2               trace.from_MG01.duration_p95     31  1.025025e+00
19   trace  dockerA2                  trace.from_MG01.row_count     31  1.390000e+02,      cmdb_id
0       IG01
1       IG02
2       MG01
3       MG02
4    Mysql01
..       ...
24  apache02
25  dockerA1
26  dockerA2
27  dockerB1
28  dockerB2

[29 rows x 1 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse metric_df if available in the kernel; otherwise load from CSV
if 'metric_df' not in globals():
    metric_df = pd.read_csv("metric.csv")

# Ensure timestamp is timezone-aware in UTC+8 (Asia/Shanghai)
if 'timestamp' in metric_df.columns:
    metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Determine KPI column name in metric_df
if 'kpi_name' in metric_df.columns:
    kpi_col = 'kpi_name'
elif 'KPI_name' in metric_df.columns:
    kpi_col = 'KPI_name'
else:
    raise KeyError("No kpi_name / KPI_name column found in metric_df")

# Ensure 'value' is numeric
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

# Aggregate by (cmdb_id, kpi_name): count and global P95 (using all rows in metric_df)
metric_agg = (
    metric_df
    .groupby(['cmdb_id', kpi_col])['value']
    .agg(count='count', p95_value=lambda x: x.quantile(0.95))
    .reset_index()
)

# Standardize column name to 'kpi_name' for output
if kpi_col != 'kpi_name':
    metric_agg = metric_agg.rename(columns={kpi_col: 'kpi_name'})

# Top 20 groups sorted by count descending (limit to at most 20 rows)
metric_top20 = metric_agg.sort_values(by='count', ascending=False).head(20).reset_index(drop=True)
metric_top20 = metric_top20[['cmdb_id', 'kpi_name', 'count', 'p95_value']]

# Full list of unique cmdb_id values present in metric.csv
unique_cmdbs_metric = pd.DataFrame(sorted(metric_df['cmdb_id'].dropna().unique()), columns=['cmdb_id'])

# Display results (as variables, per instructions)
metric_top20, unique_cmdbs_metric

```
Out[2]:
```
```python
# Summarize the metric aggregation results in plain English using existing variables.
# Reuse metric_top20 and unique_cmdbs_metric from the kernel.

num_unique = int(unique_cmdbs_metric.shape[0])
top_cmdb_counts = metric_top20['cmdb_id'].value_counts().head(5).to_dict()
example_cmdbs = ', '.join(unique_cmdbs_metric['cmdb_id'].head(6).tolist())

summary = (
    "Summary:\n"
    f"- The top 20 (cmdb_id, kpi_name) groups in metric.csv all have count = {int(metric_top20['count'].iloc[0])} data points.\n"
    f"- Most of the top groups come from Tomcat04 (several distinct KPIs); IG01 also appears among the top entries.\n"
    f"- Number of unique cmdb_id values in metric.csv: {num_unique}.\n"
    f"- Example cmdb_ids: {example_cmdbs}.\n"
    f"- Top cmdb_id frequency in the top-20: {top_cmdb_counts}."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id                                           kpi_name  count     p95_value
0   Tomcat03  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...     31  0.000000e+00
1       IG01           JVM-Threads_7778_JVM_ThreadCount_Threads     31  7.500000e+01
2   Tomcat04  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...     31  2.000000e+00
3   Tomcat04  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...     31  5.000000e+02
4   Tomcat04     Tomcat-Sessions_7441--_SESSIONRejectedSessions     31  0.000000e+00
5   Tomcat04  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...     31  0.000000e+00
6   Tomcat04  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...     31  0.000000e+00
7   Tomcat04  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...     31  0.000000e+00
8   Tomcat04  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...     31  0.000000e+00
9   Tomcat04  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...     31  0.000000e+00
10  Tomcat04  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...     31  0.000000e+00
11  Tomcat04  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...     31  1.460000e+02
12  Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...     31  4.136860e+05
13  Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_Processin...     31  1.317936e+09
14  Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_RequestCo...     31  2.237617e+06
15  Tomcat04    Tomcat-Sessions_7441--UOCP_SESSIONActiveCounter     31  0.000000e+00
16  Tomcat04  Tomcat-Sessions_7441--UOCP_SESSIONKeepaliveCou...     31  0.000000e+00
17  Tomcat04  Tomcat-Sessions_7441--UOCP_SESSIONRejectedSess...     31  0.000000e+00
18  Tomcat04        Tomcat-Sessions_7441--_SESSIONActiveCounter     31  0.000000e+00
19  Tomcat04     Tomcat-Sessions_7441--_SESSIONKeepaliveCounter     31  0.000000e+00,      cmdb_id
0       IG01
1       IG02
2       MG01
3       MG02
4    Mysql01
..       ...
21  Tomcat03
22  Tomcat04
23  apache01
24  apache02
25  dockerA2

[26 rows x 1 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse metric_df if available; otherwise load
if 'metric_df' not in globals():
    metric_df = pd.read_csv("metric.csv")

# Ensure timestamp is timezone-aware in UTC+8
if 'timestamp' in metric_df.columns:
    # If integer/float epoch seconds -> convert; if already datetime but naive -> localize; if tz-aware convert
    if pd.api.types.is_integer_dtype(metric_df['timestamp']) or pd.api.types.is_float_dtype(metric_df['timestamp']):
        metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    else:
        metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], utc=True, errors='coerce')
        # If after parsing it's tz-aware, convert; if it's naive, localize to UTC then convert
        if metric_df['timestamp'].dt.tz is None:
            metric_df['timestamp'] = metric_df['timestamp'].dt.tz_localize('UTC').dt.tz_convert(tz)
        else:
            metric_df['timestamp'] = metric_df['timestamp'].dt.tz_convert(tz)

# Determine kpi column name
if 'kpi_name' in metric_df.columns:
    kpi_col = 'kpi_name'
elif 'KPI_name' in metric_df.columns:
    kpi_col = 'KPI_name'
else:
    raise KeyError("metric_df has no kpi_name / KPI_name column")

# Ensure numeric 'value'
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

# Compute global P95 per (cmdb_id, kpi_name) using all rows (global thresholds)
global_p95 = (
    metric_df
    .groupby(['cmdb_id', kpi_col])['value']
    .agg(global_p95=lambda x: x.quantile(0.95), group_count='count')
    .reset_index()
)

# Total number of metric groups scanned
total_groups_scanned = int(global_p95.shape[0])

# Define incident window in Asia/Shanghai (UTC+8)
start_window = tz.localize(pd.Timestamp("2021-03-24 19:30:00"))
end_window = tz.localize(pd.Timestamp("2021-03-24 20:00:00"))

# Filter metric_df to incident window (inclusive start, exclusive end)
window_df = metric_df[
    (metric_df['timestamp'] >= start_window) &
    (metric_df['timestamp'] < end_window)
].copy()

# Merge global_p95 into windowed rows to compare
window_with_p95 = window_df.merge(global_p95, on=['cmdb_id', kpi_col], how='left')

# Identify anomaly rows where value >= global_p95 (note: if global_p95 is NaN, comparison yields False)
anomaly_rows = window_with_p95[window_with_p95['value'] >= window_with_p95['global_p95']].copy()

# Aggregate anomaly rows per group
if not anomaly_rows.empty:
    agg = anomaly_rows.groupby(['cmdb_id', kpi_col]).agg(
        count_anomalies=('value', 'count'),
        earliest_anomaly_timestamp=('timestamp', 'min'),
        max_value_in_window=('value', 'max'),
        global_p95=('global_p95', 'first')
    ).reset_index()

    # Compute severity_ratio, handle division by zero
    agg['severity_ratio'] = agg.apply(
        lambda r: np.inf if (r['global_p95'] == 0 and r['max_value_in_window'] > 0) else
                  (np.nan if r['global_p95'] == 0 and r['max_value_in_window'] == 0 else
                   r['max_value_in_window'] / r['global_p95']),
        axis=1
    )

    # Format earliest_anomaly_timestamp as timezone-aware strings in UTC+8 for compact display
    agg['earliest_anomaly_timestamp'] = agg['earliest_anomaly_timestamp'].dt.tz_convert(tz)

    # Select and sort by severity_ratio descending, limit to top 20
    anomalies_summary_top = (
        agg[['cmdb_id', kpi_col, 'global_p95', 'count_anomalies', 'earliest_anomaly_timestamp', 'max_value_in_window', 'severity_ratio']]
        .sort_values(by='severity_ratio', ascending=False)
        .head(20)
        .reset_index(drop=True)
    )
else:
    # Empty result scaffold
    anomalies_summary_top = pd.DataFrame(columns=[
        'cmdb_id', kpi_col, 'global_p95', 'count_anomalies', 'earliest_anomaly_timestamp', 'max_value_in_window', 'severity_ratio'
    ])

# Number of groups that had anomalies inside the window
groups_with_anomalies_count = int(anomalies_summary_top.shape[0])

# Also provide count of distinct groups in anomaly_rows (in case multiple groups but fewer than top selection)
if not anomaly_rows.empty:
    groups_with_anomalies_count_full = int(anomaly_rows.groupby(['cmdb_id', kpi_col]).ngroups)
else:
    groups_with_anomalies_count_full = 0

# Final returned variables (compact): anomalies summary (top <=20), total groups scanned, groups with anomalies (full count)
anomalies_summary_top, total_groups_scanned, groups_with_anomalies_count_full

```
Out[3]:
```
```python
# Summarize anomaly scan results in plain English using existing variables from the kernel.
# Reuse: anomalies_summary_top, total_groups_scanned, groups_with_anomalies_count_full

num_scanned = int(total_groups_scanned)
num_anomalous = int(groups_with_anomalies_count_full)

# Prepare brief examples from top anomalies (up to 5)
examples = []
for _, r in anomalies_summary_top.head(5).iterrows():
    sev = r['severity_ratio']
    sev_str = "inf" if pd.isna(sev) and r['global_p95']==0 and r['max_value_in_window']>0 else (
              "NaN" if pd.isna(sev) else f"{float(sev):.3g}")
    examples.append(
        f"{r['cmdb_id']} / {r['KPI_name']}: anomalies={int(r['count_anomalies'])}, "
        f"earliest={pd.Timestamp(r['earliest_anomaly_timestamp']).strftime('%Y-%m-%d %H:%M:%S%z')}, "
        f"max={float(r['max_value_in_window']):.6g}, severity_ratio={sev_str}"
    )

summary = (
    f"Summary of metric anomaly scan (incident window 2021-03-24 19:30 to 20:00 UTC+8):\n"
    f"- Total metric groups scanned: {num_scanned}.\n"
    f"- Groups with >=1 anomaly inside the window: {num_anomalous}.\n"
    f"- Top groups by severity_ratio (showing up to 5 examples):\n  - " + "\n  - ".join(examples) + "\n"
    f"- Notes: Several groups (e.g., Mysql02 and Mysql01 metrics) have global P95 = 0 while values >0 occurred in the window, producing infinite severity ratios. Other high-severity items include disk and CPU I/O related KPIs on MG01/MG02/Tomcat nodes."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(         cmdb_id                                           KPI_name  global_p95  count_anomalies earliest_anomaly_timestamp  max_value_in_window  severity_ratio
0        Mysql02          Mysql-MySQL_3306_CurrentSQLMaxRunningTime    0.000000               30  2021-03-24 19:30:00+08:00             2.000000             inf
1        Mysql02        Mysql-MySQL_3306_Innodb data pending writes    0.000000               30  2021-03-24 19:30:00+08:00             1.000000             inf
2        Mysql01        Mysql-MySQL_3306_Innodb data pending fsyncs    0.000000               30  2021-03-24 19:30:00+08:00             1.000000             inf
3       apache01     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT    0.100000                1  2021-03-24 19:53:00+08:00             1.000000       10.000000
4        Redis02                             OSLinux-CPU_CPU_CPUWio    0.095295                2  2021-03-24 19:35:00+08:00             0.484039        5.079375
5           MG02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead   24.644075                1  2021-03-24 19:50:00+08:00            98.576300        4.000000
6           MG02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps    0.067800                1  2021-03-24 19:50:00+08:00             0.271200        4.000000
7       Tomcat01                             OSLinux-CPU_CPU_CPUWio    0.275264                2  2021-03-24 19:45:00+08:00             1.089500        3.958016
8       Tomcat01     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps    0.300000                1  2021-03-24 19:41:00+08:00             1.000000        3.333333
9        Redis02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKPer...    2.520000                1  2021-03-24 19:35:00+08:00             8.400000        3.333333
10          MG01     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps    0.300000                1  2021-03-24 19:50:00+08:00             1.000000        3.333333
11      Tomcat03     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps    3.250000                1  2021-03-24 19:42:00+08:00            10.000000        3.076923
12      Tomcat04                             OSLinux-CPU_CPU_CPUWio    0.407290                2  2021-03-24 19:33:00+08:00             1.189436        2.920366
13      Tomcat01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKPer...    0.105000                1  2021-03-24 19:45:00+08:00             0.300000        2.857143
14      Tomcat04     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps    2.500000                1  2021-03-24 19:50:00+08:00             7.000000        2.800000
15      Tomcat04  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...    2.000000                4  2021-03-24 19:37:00+08:00             5.000000        2.500000
16  ServiceTest6                                            app.mrt    0.844845                2  2021-03-24 19:45:00+08:00             2.096500        2.481520
17      apache02                             OSLinux-CPU_CPU_CPUWio    0.367705                1  2021-03-24 19:45:00+08:00             0.880800        2.395398
18          IG02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKTps    0.450000                1  2021-03-24 19:35:00+08:00             1.000000        2.222222
19      Tomcat01     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKTps    0.450000                1  2021-03-24 19:51:00+08:00             1.000000        2.222222, 1756, 1698)```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse trace_df if available; otherwise load from CSV
if 'trace_df' not in globals():
    trace_df = pd.read_csv("trace.csv")

# Ensure timestamp is timezone-aware in UTC+8
if 'timestamp' in trace_df.columns:
    if pd.api.types.is_integer_dtype(trace_df['timestamp']) or pd.api.types.is_float_dtype(trace_df['timestamp']):
        trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    else:
        trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], utc=True, errors='coerce')
        if trace_df['timestamp'].dt.tz is None:
            trace_df['timestamp'] = trace_df['timestamp'].dt.tz_localize('UTC').dt.tz_convert(tz)
        else:
            trace_df['timestamp'] = trace_df['timestamp'].dt.tz_convert(tz)

# Determine trace name column
if 'trace_name' in trace_df.columns:
    trace_col = 'trace_name'
elif 'trace_name' not in trace_df.columns and 'KPI_name' in trace_df.columns:
    # earlier code might have renamed to KPI_name
    trace_col = 'KPI_name'
else:
    raise KeyError("trace_df has no trace_name / KPI_name column")

# Ensure numeric 'value'
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')

# Compute global P95 per (cmdb_id, trace_name) using all rows
global_p95_trace = (
    trace_df
    .groupby(['cmdb_id', trace_col])['value']
    .agg(global_p95=lambda x: x.quantile(0.95))
    .reset_index()
)

# Total number of trace groups scanned
total_groups_scanned_trace = int(global_p95_trace.shape[0])

# Define incident window in Asia/Shanghai (UTC+8)
start_window = tz.localize(pd.Timestamp("2021-03-24 19:30:00"))
end_window = tz.localize(pd.Timestamp("2021-03-24 20:00:00"))

# Filter trace_df to incident window (inclusive start, exclusive end)
trace_window = trace_df[
    (trace_df['timestamp'] >= start_window) &
    (trace_df['timestamp'] < end_window)
].copy()

# Merge global_p95 into windowed rows
trace_window = trace_window.merge(global_p95_trace, on=['cmdb_id', trace_col], how='left')

# Identify anomaly rows where value >= global_p95
anomaly_rows_trace = trace_window[trace_window['value'] >= trace_window['global_p95']].copy()

# Aggregate anomaly rows per group
if not anomaly_rows_trace.empty:
    agg_trace = anomaly_rows_trace.groupby(['cmdb_id', trace_col]).agg(
        count_anomalies=('value', 'count'),
        earliest_anomaly_timestamp=('timestamp', 'min'),
        max_value_in_window=('value', 'max'),
        global_p95=('global_p95', 'first')
    ).reset_index()

    # Compute severity_ratio with handling for zero global_p95
    def compute_sev(row):
        gp = row['global_p95']
        mv = row['max_value_in_window']
        if pd.isna(gp):
            return np.nan
        if gp == 0:
            if mv > 0:
                return np.inf
            if mv == 0:
                return np.nan
        return mv / gp

    agg_trace['severity_ratio'] = agg_trace.apply(compute_sev, axis=1)

    # Ensure earliest_anomaly_timestamp is in UTC+8 tz (already should be)
    agg_trace['earliest_anomaly_timestamp'] = agg_trace['earliest_anomaly_timestamp'].dt.tz_convert(tz)

    # Select and sort top groups by severity_ratio descending, limit to 20
    anomalies_trace_top20 = (
        agg_trace[['cmdb_id', trace_col, 'global_p95', 'count_anomalies', 'earliest_anomaly_timestamp', 'max_value_in_window', 'severity_ratio']]
        .sort_values(by='severity_ratio', ascending=False)
        .head(20)
        .reset_index(drop=True)
    )
else:
    anomalies_trace_top20 = pd.DataFrame(columns=[
        'cmdb_id', trace_col, 'global_p95', 'count_anomalies', 'earliest_anomaly_timestamp', 'max_value_in_window', 'severity_ratio'
    ])

# Number of trace groups that had anomalies (distinct groups)
if not anomaly_rows_trace.empty:
    groups_with_anomalies_count_full = int(anomaly_rows_trace.groupby(['cmdb_id', trace_col]).ngroups)
else:
    groups_with_anomalies_count_full = 0

# Return compact outputs
anomalies_trace_top20, total_groups_scanned_trace, groups_with_anomalies_count_full

```
Out[4]:
```
```python
# Summarize the trace anomaly scan results in plain English using existing kernel variables.
# Reuse: anomalies_trace_top20, total_groups_scanned_trace, groups_with_anomalies_count_full

num_scanned = int(total_groups_scanned_trace)
num_anomalous = int(groups_with_anomalies_count_full)

# Prepare up to 5 example rows from the top anomalies table
examples = []
for _, r in anomalies_trace_top20.head(5).iterrows():
    cmdb = r['cmdb_id']
    name = r['KPI_name']
    gp = r['global_p95']
    cnt = int(r['count_anomalies'])
    ts = pd.Timestamp(r['earliest_anomaly_timestamp']).strftime('%Y-%m-%d %H:%M:%S%z')
    mv = float(r['max_value_in_window'])
    sev = r['severity_ratio']
    sev_str = "inf" if np.isinf(sev) else (f"{float(sev):.3f}" if not pd.isna(sev) else "NaN")
    examples.append(f"{cmdb} / {name}: anomalies={cnt}, earliest={ts}, max={mv:.6g}, severity_ratio={sev_str}")

summary = (
    f"Trace anomaly scan summary (window 2021-03-24 19:30–20:00 UTC+8):\n"
    f"- Total trace groups scanned: {num_scanned}.\n"
    f"- Groups with ≥1 anomaly inside the window: {num_anomalous}.\n"
    f"- Top groups by severity_ratio (examples, up to 5):\n  - " + "\n  - ".join(examples) + "\n"
    f"- Notes: Highest severity ratios are for latency-type traces (e.g., trace.self.duration_p95 on dockerB1 and Tomcat04), indicating large spikes relative to their global P95 baselines."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id                           KPI_name  global_p95  count_anomalies earliest_anomaly_timestamp  max_value_in_window  severity_ratio
0   dockerB1            trace.self.duration_p95    0.019800                2  2021-03-24 19:45:00+08:00             0.225000       11.363636
1   Tomcat04            trace.self.duration_p95    0.054500                2  2021-03-24 19:51:00+08:00             0.499500        9.165138
2       MG02  trace.from_dockerB1.duration_mean    0.461418                2  2021-03-24 19:51:00+08:00             3.459688        7.497940
3   dockerB1        trace.to_MG02.duration_mean    0.461418                2  2021-03-24 19:51:00+08:00             3.459688        7.497940
4   Tomcat03            trace.self.duration_p95    0.148800                2  2021-03-24 19:52:00+08:00             0.666150        4.476815
5       MG02  trace.from_Tomcat01.duration_mean    0.372900                2  2021-03-24 19:45:00+08:00             1.397022        3.746372
6   Tomcat01        trace.to_MG02.duration_mean    0.372900                2  2021-03-24 19:45:00+08:00             1.397022        3.746372
7   Tomcat02         trace.to_MG02.duration_p95    2.278975                2  2021-03-24 19:51:00+08:00             8.193100        3.595081
8       MG02   trace.from_Tomcat02.duration_p95    2.278975                2  2021-03-24 19:51:00+08:00             8.193100        3.595081
9       MG02   trace.from_Tomcat01.duration_p95    2.533725                2  2021-03-24 19:45:00+08:00             8.371750        3.304127
10  Tomcat01         trace.to_MG02.duration_p95    2.533725                2  2021-03-24 19:45:00+08:00             8.371750        3.304127
11      MG01   trace.from_dockerB1.duration_p95    2.097500                2  2021-03-24 19:45:00+08:00             6.269500        2.989035
12  dockerB1         trace.to_MG01.duration_p95    2.097500                2  2021-03-24 19:45:00+08:00             6.269500        2.989035
13  Tomcat01       trace.from_IG02.duration_p95    2.716900                2  2021-03-24 19:45:00+08:00             8.055050        2.964794
14      IG02     trace.to_Tomcat01.duration_p95    2.716900                2  2021-03-24 19:45:00+08:00             8.055050        2.964794
15  Tomcat02        trace.to_MG02.duration_mean    0.395910                2  2021-03-24 19:51:00+08:00             1.134051        2.864420
16      MG02  trace.from_Tomcat02.duration_mean    0.395910                2  2021-03-24 19:51:00+08:00             1.134051        2.864420
17      MG01            trace.self.duration_p95    1.117725                2  2021-03-24 19:45:00+08:00             3.126000        2.796752
18      MG02  trace.from_dockerA1.duration_mean    0.317787                2  2021-03-24 19:43:00+08:00             0.862571        2.714306
19  dockerA1        trace.to_MG02.duration_mean    0.317787                2  2021-03-24 19:43:00+08:00             0.862571        2.714306, 264, 264)```
```

In [ ]:
# In[5]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse metric_df if available; otherwise load
if 'metric_df' not in globals():
    metric_df = pd.read_csv("metric.csv")

# Ensure timestamp is timezone-aware in UTC+8
if 'timestamp' in metric_df.columns:
    if pd.api.types.is_integer_dtype(metric_df['timestamp']) or pd.api.types.is_float_dtype(metric_df['timestamp']):
        metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    else:
        metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], utc=True, errors='coerce')
        if metric_df['timestamp'].dt.tz is None:
            metric_df['timestamp'] = metric_df['timestamp'].dt.tz_localize('UTC').dt.tz_convert(tz)
        else:
            metric_df['timestamp'] = metric_df['timestamp'].dt.tz_convert(tz)

# Determine KPI column name and standardize to 'kpi_name' for output
if 'kpi_name' in metric_df.columns:
    kpi_col = 'kpi_name'
elif 'KPI_name' in metric_df.columns:
    kpi_col = 'KPI_name'
else:
    raise KeyError("metric_df has no kpi_name / KPI_name column")

# Define incident window in UTC+8 (Asia/Shanghai)
start_window = tz.localize(pd.Timestamp("2021-03-24 19:30:00"))
end_window = tz.localize(pd.Timestamp("2021-03-24 20:00:00"))

# Filter by time window and by cmdb_id in Mysql01/Mysql02
target_cmdbs = ['Mysql01', 'Mysql02']
metric_window = metric_df[
    (metric_df['timestamp'] >= start_window) &
    (metric_df['timestamp'] < end_window) &
    (metric_df['cmdb_id'].isin(target_cmdbs))
].copy()

# Select and rename columns for output
output_cols = ['timestamp', 'cmdb_id', kpi_col, 'value']
metric_window = metric_window.loc[:, output_cols]
if kpi_col != 'kpi_name':
    metric_window = metric_window.rename(columns={kpi_col: 'kpi_name'})

# Sort by cmdb_id then timestamp ascending
metric_window = metric_window.sort_values(by=['cmdb_id', 'timestamp'], ascending=[True, True])

# Limit to first 100 rows for display
metric_window_head = metric_window.head(100).reset_index(drop=True)

# Summary counts per target cmdb_id (ensure both keys present)
counts = metric_window.groupby('cmdb_id').size().reindex(target_cmdbs, fill_value=0).reset_index()
counts.columns = ['cmdb_id', 'row_count']

# Return compact results: filtered rows (up to 100) and counts per cmdb_id
metric_window_head, counts

```
Out[5]:
```
```python
# Summarize the filtered metric rows for Mysql01 and Mysql02 in plain English.
# Reuse existing variables: metric_window_head, counts

num_displayed = int(metric_window_head.shape[0])
total_rows_filtered = int(counts['row_count'].sum())
mysql01_count = int(counts.loc[counts['cmdb_id']=='Mysql01', 'row_count'].iloc[0])
mysql02_count = int(counts.loc[counts['cmdb_id']=='Mysql02', 'row_count'].iloc[0])

summary = (
    f"Summary:\n"
    f"- Returned table: up to {num_displayed} rows from metric.csv filtered to 2021-03-24 19:30–20:00 (UTC+8) and cmdb_id in {{'Mysql01','Mysql02'}}, "
    f"with columns (timestamp, cmdb_id, kpi_name, value), sorted by cmdb_id then timestamp ascending.\n"
    f"- Total matching rows in that window: {total_rows_filtered}.\n"
    f"- Rows per cmdb_id: Mysql01 = {mysql01_count}, Mysql02 = {mysql02_count}.\n"
    f"- Note: The displayed table shows the first {num_displayed} rows (Mysql01 rows appear first)."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                   timestamp  cmdb_id                                     kpi_name    value
0  2021-03-24 19:30:00+08:00  Mysql01             Mysql-MySQL_3306_Aborted Clients   0.0000
1  2021-03-24 19:30:00+08:00  Mysql01            Mysql-MySQL_3306_Aborted Connects   0.0000
2  2021-03-24 19:30:00+08:00  Mysql01       Mysql-MySQL_3306_Binlog Cache Disk Use   0.0000
3  2021-03-24 19:30:00+08:00  Mysql01            Mysql-MySQL_3306_Binlog Cache Use   1.4333
4  2021-03-24 19:30:00+08:00  Mysql01  Mysql-MySQL_3306_Binlog stmt cache disk use   0.0000
..                       ...      ...                                          ...      ...
95 2021-03-24 19:30:00+08:00  Mysql01           Mysql-MySQL_3306_Qcache Not Cached  20.3167
96 2021-03-24 19:30:00+08:00  Mysql01     Mysql-MySQL_3306_Qcache Queries In Cache   0.0000
97 2021-03-24 19:30:00+08:00  Mysql01         Mysql-MySQL_3306_Qcache Total Blocks   1.0000
98 2021-03-24 19:30:00+08:00  Mysql01                   Mysql-MySQL_3306_Questions  25.1500
99 2021-03-24 19:30:00+08:00  Mysql01                   Mysql-MySQL_3306_Rows Read  63.9000

[100 rows x 4 columns],    cmdb_id  row_count
0  Mysql01       4642
1  Mysql02       4621)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[6]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse metric_df if present; otherwise load
if 'metric_df' not in globals():
    metric_df = pd.read_csv("metric.csv")

# Ensure timestamp is timezone-aware in UTC+8
if 'timestamp' in metric_df.columns:
    if pd.api.types.is_integer_dtype(metric_df['timestamp']) or pd.api.types.is_float_dtype(metric_df['timestamp']):
        metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    else:
        metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], utc=True, errors='coerce')
        if metric_df['timestamp'].dt.tz is None:
            metric_df['timestamp'] = metric_df['timestamp'].dt.tz_localize('UTC').dt.tz_convert(tz)
        else:
            metric_df['timestamp'] = metric_df['timestamp'].dt.tz_convert(tz)

# Determine KPI column name
if 'kpi_name' in metric_df.columns:
    kpi_col = 'kpi_name'
elif 'KPI_name' in metric_df.columns:
    kpi_col = 'KPI_name'
else:
    raise KeyError("metric_df has no kpi_name / KPI_name column")

# Ensure numeric 'value'
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

# 1) Compute global P95 per (cmdb_id, kpi_name) using all rows (global thresholds)
global_p95 = (
    metric_df
    .groupby(['cmdb_id', kpi_col])['value']
    .agg(global_p95=lambda x: x.quantile(0.95))
    .reset_index()
)

# 2) Filter to incident window and target cmdb_ids
start_window = tz.localize(pd.Timestamp("2021-03-24 19:30:00"))
end_window = tz.localize(pd.Timestamp("2021-03-24 20:00:00"))

target_cmdbs = ['Mysql01', 'Mysql02']

window_df = metric_df[
    (metric_df['timestamp'] >= start_window) &
    (metric_df['timestamp'] < end_window) &
    (metric_df['cmdb_id'].isin(target_cmdbs))
].copy()

# 3) Merge global p95 into windowed rows (thresholds computed globally)
window_with_p95 = window_df.merge(global_p95, on=['cmdb_id', kpi_col], how='left')

# 4) Identify anomaly rows:
# - If global_p95 > 0: anomaly when value >= global_p95
# - If global_p95 == 0: anomaly when value > 0
cond_nonzero = window_with_p95['global_p95'] > 0
cond_zero = window_with_p95['global_p95'] == 0

anomaly_mask = (cond_nonzero & (window_with_p95['value'] >= window_with_p95['global_p95'])) | \
               (cond_zero & (window_with_p95['value'] > 0))

anomaly_rows = window_with_p95[anomaly_mask].copy()

# 5) Aggregate anomaly rows per group
if not anomaly_rows.empty:
    agg = anomaly_rows.groupby(['cmdb_id', kpi_col]).agg(
        count_anomalies=('value', 'count'),
        earliest_anomaly_timestamp=('timestamp', 'min'),
        max_value_in_window=('value', 'max'),
        global_p95=('global_p95', 'first')
    ).reset_index()

    # Compute severity_ratio with specified handling
    def compute_severity(row):
        gp = row['global_p95']
        mv = row['max_value_in_window']
        if pd.isna(gp):
            return np.nan
        if gp == 0:
            if mv > 0:
                return np.inf
            else:
                return np.nan
        return mv / gp

    agg['severity_ratio'] = agg.apply(compute_severity, axis=1)

    # Ensure earliest timestamp is timezone-aware in UTC+8
    agg['earliest_anomaly_timestamp'] = agg['earliest_anomaly_timestamp'].dt.tz_convert(tz)

    # Standardize column name to kpi_name for output if needed
    if kpi_col != 'kpi_name':
        agg = agg.rename(columns={kpi_col: 'kpi_name'})
    else:
        agg = agg.rename(columns={kpi_col: 'kpi_name'})

    # Select and sort: count_anomalies desc, then severity_ratio desc; limit to top 20
    anomalies_summary_top20 = (
        agg[['cmdb_id', 'kpi_name', 'global_p95', 'count_anomalies', 'earliest_anomaly_timestamp', 'max_value_in_window', 'severity_ratio']]
        .sort_values(by=['count_anomalies', 'severity_ratio'], ascending=[False, False])
        .head(20)
        .reset_index(drop=True)
    )

    # 6) KPIs anomalous for every minute of the 30-minute window (count_anomalies == 30)
    kpis_all_30 = (agg[agg['count_anomalies'] == 30][['kpi_name']].drop_duplicates().reset_index(drop=True))
else:
    anomalies_summary_top20 = pd.DataFrame(columns=['cmdb_id', 'kpi_name', 'global_p95', 'count_anomalies', 'earliest_anomaly_timestamp', 'max_value_in_window', 'severity_ratio'])
    kpis_all_30 = pd.DataFrame(columns=['kpi_name'])

# Return compact outputs: top anomalies (<=20) and kpi list with count==30
anomalies_summary_top20, kpis_all_30

```
Out[6]:
```
```python
# Summarize the previous anomaly aggregation results for Mysql01/Mysql02 in plain English.
# Reuse existing variables from the kernel: anomalies_summary_top20, kpis_all_30, possibly agg

import pandas as pd
import numpy as np

# Determine total groups that had anomalies (use agg if available for full count, otherwise fallback to anomalies_summary_top20)
if 'agg' in globals():
    groups_with_anomalies_count = int(agg.shape[0])
else:
    groups_with_anomalies_count = int(anomalies_summary_top20.shape[0])

# Number of KPIs anomalous for every minute (count_anomalies == 30)
num_kpis_all_30 = int(kpis_all_30.shape[0])

# Example KPI names (show up to 10) that were anomalous for all 30 minutes
example_kpis = kpis_all_30['kpi_name'].head(10).tolist()

summary = (
    "Summary of metric anomaly results for Mysql01 and Mysql02 (window 2021-03-24 19:30–20:00 UTC+8):\n"
    f"- Total distinct (cmdb_id, kpi_name) groups with anomalies in the window: {groups_with_anomalies_count}.\n"
    f"- Number of KPI names that were anomalous for every minute of the 30-minute window (count_anomalies == 30): {num_kpis_all_30}.\n"
    f"- Many top groups have count_anomalies = 30 and severity_ratio = 1.0 (i.e., window values equal their global P95 baselines).\n"
    f"- Example KPIs anomalous for all 30 minutes (showing up to 10): {example_kpis}.\n"
    "- Interpretation: Both Mysql01 and Mysql02 exhibit multiple KPIs that were continuously at or above their global P95 during the incident window; several metrics had values exactly equal to their global P95 (severity_ratio = 1.0)."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(    cmdb_id                                        kpi_name  global_p95  count_anomalies earliest_anomaly_timestamp  max_value_in_window  severity_ratio
0   Mysql01          Mysql-MySQL_3306_Binlog stmt cache use       360.0               30  2021-03-24 19:30:00+08:00                360.0             1.0
1   Mysql01      Mysql-MySQL_3306_GetConnectedStateOfMysqld         1.0               30  2021-03-24 19:30:00+08:00                  1.0             1.0
2   Mysql01  Mysql-MySQL_3306_Innodb buffer pool pages free    252197.0               30  2021-03-24 19:30:00+08:00             252197.0             1.0
3   Mysql01         Mysql-MySQL_3306_Innodb open files  num       165.0               30  2021-03-24 19:30:00+08:00                165.0             1.0
4   Mysql01       Mysql-MySQL_3306_Innodb row lock time avg       336.0               30  2021-03-24 19:30:00+08:00                336.0             1.0
5   Mysql01       Mysql-MySQL_3306_Innodb row lock time max     31867.0               30  2021-03-24 19:30:00+08:00              31867.0             1.0
6   Mysql01           Mysql-MySQL_3306_Max Used Connections       108.0               30  2021-03-24 19:30:00+08:00                108.0             1.0
7   Mysql01                 Mysql-MySQL_3306_MaxConnections      2000.0               30  2021-03-24 19:30:00+08:00               2000.0             1.0
8   Mysql01                     Mysql-MySQL_3306_Open Files       207.0               30  2021-03-24 19:30:00+08:00                207.0             1.0
9   Mysql01                    Mysql-MySQL_3306_Open Tables      4569.0               30  2021-03-24 19:30:00+08:00               4569.0             1.0
10  Mysql01             Mysql-MySQL_3306_Qcache Free Blocks         1.0               30  2021-03-24 19:30:00+08:00                  1.0             1.0
11  Mysql01             Mysql-MySQL_3306_Qcache Free Memory   1031832.0               30  2021-03-24 19:30:00+08:00            1031832.0             1.0
12  Mysql01            Mysql-MySQL_3306_Qcache Total Blocks         1.0               30  2021-03-24 19:30:00+08:00                  1.0             1.0
13  Mysql01                      Mysql-MySQL_3306_Sort Rows   1498608.0               30  2021-03-24 19:30:00+08:00            1498608.0             1.0
14  Mysql01                 Mysql-MySQL_3306_Threads Cached        65.0               30  2021-03-24 19:30:00+08:00                 65.0             1.0
15  Mysql01               Mysql-MySQL_3306_ThreadsConnected        43.0               30  2021-03-24 19:30:00+08:00                 43.0             1.0
16  Mysql01                 Mysql-MySQL_3306_ThreadsRunning         1.0               30  2021-03-24 19:30:00+08:00                  1.0             1.0
17  Mysql02          Mysql-MySQL_3306_Binlog stmt cache use      4708.0               30  2021-03-24 19:30:00+08:00               4708.0             1.0
18  Mysql02      Mysql-MySQL_3306_GetConnectedStateOfMysqld         1.0               30  2021-03-24 19:30:00+08:00                  1.0             1.0
19  Mysql02         Mysql-MySQL_3306_Innodb open files  num        43.0               30  2021-03-24 19:30:00+08:00                 43.0             1.0,                                           kpi_name
0           Mysql-MySQL_3306_Binlog stmt cache use
1       Mysql-MySQL_3306_GetConnectedStateOfMysqld
2   Mysql-MySQL_3306_Innodb buffer pool pages free
3          Mysql-MySQL_3306_Innodb open files  num
4        Mysql-MySQL_3306_Innodb row lock time avg
5        Mysql-MySQL_3306_Innodb row lock time max
6            Mysql-MySQL_3306_Max Used Connections
7                  Mysql-MySQL_3306_MaxConnections
8                      Mysql-MySQL_3306_Open Files
9                     Mysql-MySQL_3306_Open Tables
10             Mysql-MySQL_3306_Qcache Free Blocks
11             Mysql-MySQL_3306_Qcache Free Memory
12            Mysql-MySQL_3306_Qcache Total Blocks
13                      Mysql-MySQL_3306_Sort Rows
14                 Mysql-MySQL_3306_Threads Cached
15               Mysql-MySQL_3306_ThreadsConnected
16                 Mysql-MySQL_3306_ThreadsRunning)```
```

In [ ]:
# In[7]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse error_logs_df if present; otherwise load
if 'error_logs_df' not in globals():
    error_logs_df = pd.read_csv("error_logs.csv")

# Ensure timestamp is timezone-aware in UTC+8 (Asia/Shanghai)
if 'timestamp' in error_logs_df.columns:
    if pd.api.types.is_integer_dtype(error_logs_df['timestamp']) or pd.api.types.is_float_dtype(error_logs_df['timestamp']):
        error_logs_df['timestamp'] = pd.to_datetime(error_logs_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    else:
        error_logs_df['timestamp'] = pd.to_datetime(error_logs_df['timestamp'], utc=True, errors='coerce')
        if error_logs_df['timestamp'].dt.tz is None:
            error_logs_df['timestamp'] = error_logs_df['timestamp'].dt.tz_localize('UTC').dt.tz_convert(tz)
        else:
            error_logs_df['timestamp'] = error_logs_df['timestamp'].dt.tz_convert(tz)

# Define incident window (inclusive start, exclusive end)
start_window = tz.localize(pd.Timestamp("2021-03-24 19:30:00"))
end_window = tz.localize(pd.Timestamp("2021-03-24 20:00:00"))

# Filter to window
logs_window = error_logs_df[
    (error_logs_df['timestamp'] >= start_window) &
    (error_logs_df['timestamp'] < end_window)
].copy()

# Prepare sorted window (by cmdb_id then timestamp) so 'first' message is earliest per cmdb_id
logs_window_sorted = logs_window.sort_values(by=['cmdb_id', 'timestamp'], ascending=[True, True])

# Aggregate by cmdb_id: count_of_messages, earliest_message_timestamp, example_message (first message text)
agg_by_cmdb = (
    logs_window_sorted
    .groupby('cmdb_id', as_index=False)
    .agg(
        count_of_messages=('message', 'count'),
        earliest_message_timestamp=('timestamp', 'min'),
        example_message=('message', 'first')
    )
)

# Top 20 by count_of_messages desc
top20_cmdb = agg_by_cmdb.sort_values(by='count_of_messages', ascending=False).head(20).reset_index(drop=True)

# Case-insensitive substring search for 'mysql' in filtered messages
mysql_matches = logs_window[logs_window['message'].astype(str).str.contains('mysql', case=False, na=False)].copy()
mysql_matches = mysql_matches[['timestamp', 'cmdb_id', 'message']].sort_values(by='timestamp', ascending=True).head(20).reset_index(drop=True)

# Return compact outputs (limited to 20 rows each)
top20_cmdb, mysql_matches

```
Out[7]:
```
```python
# Summarize the error log scan results in plain English using existing variables.
count_groups = int(top20_cmdb.shape[0]) if 'top20_cmdb' in globals() else 0
count_matches = int(mysql_matches.shape[0]) if 'mysql_matches' in globals() else 0

summary = (
    "Summary:\n"
    "- No error log entries were found in error_logs.csv during the incident window "
    "2021-03-24 19:30:00 to 2021-03-24 20:00:00 (UTC+8).\n"
    f"- Aggregation by cmdb_id returned {count_groups} groups (empty result), so there are no example messages to show.\n"
    f"- A case-insensitive search for 'mysql' in the filtered messages returned {count_matches} matches (none).\n"
    "- Conclusion: There are no error log events in the provided error_logs.csv for the specified incident window."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(Empty DataFrame
Columns: [cmdb_id, count_of_messages, earliest_message_timestamp, example_message]
Index: [], Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: [])```
```

In [ ]:
# In[8]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse existing metric_df and global_p95 if present; otherwise load/compute
if 'metric_df' not in globals():
    metric_df = pd.read_csv("metric.csv")

# Ensure timestamp is timezone-aware in UTC+8
if 'timestamp' in metric_df.columns:
    if pd.api.types.is_integer_dtype(metric_df['timestamp']) or pd.api.types.is_float_dtype(metric_df['timestamp']):
        metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    else:
        metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], utc=True, errors='coerce')
        if metric_df['timestamp'].dt.tz is None:
            metric_df['timestamp'] = metric_df['timestamp'].dt.tz_localize('UTC').dt.tz_convert(tz)
        else:
            metric_df['timestamp'] = metric_df['timestamp'].dt.tz_convert(tz)

# Determine KPI column name
if 'kpi_name' in metric_df.columns:
    kpi_col = 'kpi_name'
elif 'KPI_name' in metric_df.columns:
    kpi_col = 'KPI_name'
else:
    raise KeyError("metric_df missing kpi_name / KPI_name")

# Ensure numeric value
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

# Compute global P95 per (cmdb_id, kpi_name) if not already computed
if 'global_p95' not in globals():
    global_p95 = (
        metric_df
        .groupby(['cmdb_id', kpi_col])['value']
        .agg(global_p95=lambda x: x.quantile(0.95))
        .reset_index()
    )

# Define incident window and targets
start_window = tz.localize(pd.Timestamp("2021-03-24 19:30:00"))
end_window = tz.localize(pd.Timestamp("2021-03-24 20:00:00"))
targets = ['Mysql01', 'Mysql02']
expected_minutes = 30

# Filter metric_df to window and targets
window_df = metric_df[
    (metric_df['timestamp'] >= start_window) &
    (metric_df['timestamp'] < end_window) &
    (metric_df['cmdb_id'].isin(targets))
].copy()

# Merge global_p95 into windowed rows (thresholds computed globally as required)
window_with_p95 = window_df.merge(global_p95, on=['cmdb_id', kpi_col], how='left')

# Define anomaly condition:
# - if global_p95 > 0 -> anomaly when value > global_p95
# - if global_p95 == 0 -> anomaly when value > 0
cond_gp_pos = window_with_p95['global_p95'] > 0
cond_gp_zero = window_with_p95['global_p95'] == 0
anomaly_mask = (cond_gp_pos & (window_with_p95['value'] > window_with_p95['global_p95'])) | \
               (cond_gp_zero & (window_with_p95['value'] > 0))

anomaly_rows = window_with_p95[anomaly_mask].copy()

# Count total distinct KPIs scanned per cmdb_id in the window
distinct_kpis_scanned = (
    window_df.groupby('cmdb_id')[kpi_col].nunique().reindex(targets, fill_value=0).reset_index()
)
distinct_kpis_scanned.columns = ['cmdb_id', 'total_distinct_kpis_scanned_for_this_instance']

# Aggregate anomalies per group to get count_anomalies
if not anomaly_rows.empty:
    grp = anomaly_rows.groupby(['cmdb_id', kpi_col]).agg(
        count_anomalies=('value', 'count'),
        earliest_anomaly_timestamp=('timestamp', 'min'),
        max_value_in_window=('value', 'max'),
        global_p95=('global_p95', 'first')
    ).reset_index()

    # Determine groups with count_anomalies == expected_minutes
    grp_count30 = grp[grp['count_anomalies'] == expected_minutes].copy()

    # For each target cmdb_id, compute number_of_kpis_with_count30 and example kpis (up to 10)
    rows = []
    for cmdb in targets:
        # number_of_kpis_with_count30
        num_kpis_30 = int(grp_count30[grp_count30['cmdb_id'] == cmdb].shape[0])
        # example kpi names up to 10
        examples = grp_count30[grp_count30['cmdb_id'] == cmdb][kpi_col].head(10).tolist()
        examples_str = '; '.join(examples) if examples else ''
        # total distinct scanned
        total_scanned = int(distinct_kpis_scanned.loc[distinct_kpis_scanned['cmdb_id']==cmdb, 'total_distinct_kpis_scanned_for_this_instance'].iloc[0])
        rows.append({
            'cmdb_id': cmdb,
            'number_of_kpis_with_count30': num_kpis_30,
            'total_distinct_kpis_scanned_for_this_instance': total_scanned,
            'example_kpis_count30_up_to_10': examples_str
        })

    summary_df = pd.DataFrame(rows)
else:
    # No anomalies found: return zeros and empty examples
    rows = []
    for cmdb in targets:
        total_scanned = int(distinct_kpis_scanned.loc[distinct_kpis_scanned['cmdb_id']==cmdb, 'total_distinct_kpis_scanned_for_this_instance'].iloc[0])
        rows.append({
            'cmdb_id': cmdb,
            'number_of_kpis_with_count30': 0,
            'total_distinct_kpis_scanned_for_this_instance': total_scanned,
            'example_kpis_count30_up_to_10': ''
        })
    summary_df = pd.DataFrame(rows)

# Keep output compact: two rows (one per target), selected columns
summary_df = summary_df[['cmdb_id', 'number_of_kpis_with_count30', 'total_distinct_kpis_scanned_for_this_instance', 'example_kpis_count30_up_to_10']]

summary_df

```
Out[8]:
```
```python
# Summarize results in plain English using existing summary_df from the kernel.
lines = ["Summary of KPI continuous-anomaly check for the incident window (2021-03-24 19:30–20:00 UTC+8):"]
for r in summary_df.itertuples(index=False):
    examples = r.example_kpis_count30_up_to_10 if r.example_kpis_count30_up_to_10 else "None"
    lines.append(f"- {r.cmdb_id}: {r.number_of_kpis_with_count30} KPI(s) were anomalous for all 30 minutes; total distinct KPIs scanned = {r.total_distinct_kpis_scanned_for_this_instance}; example KPIs (up to 10): {examples}")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id  number_of_kpis_with_count30  total_distinct_kpis_scanned_for_this_instance example_kpis_count30_up_to_10
0  Mysql01                            0                                            223                              
1  Mysql02                            0                                            223```
```

In [ ]:
# In[9]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse metric_df if present; otherwise load
if 'metric_df' not in globals():
    metric_df = pd.read_csv("metric.csv")

# Ensure timestamp is timezone-aware in UTC+8
if 'timestamp' in metric_df.columns:
    if pd.api.types.is_integer_dtype(metric_df['timestamp']) or pd.api.types.is_float_dtype(metric_df['timestamp']):
        metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    else:
        metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], utc=True, errors='coerce')
        if metric_df['timestamp'].dt.tz is None:
            metric_df['timestamp'] = metric_df['timestamp'].dt.tz_localize('UTC').dt.tz_convert(tz)
        else:
            metric_df['timestamp'] = metric_df['timestamp'].dt.tz_convert(tz)

# Determine KPI column name and standardize to 'kpi_name'
if 'kpi_name' in metric_df.columns:
    kpi_col = 'kpi_name'
elif 'KPI_name' in metric_df.columns:
    metric_df = metric_df.rename(columns={'KPI_name': 'kpi_name'})
    kpi_col = 'kpi_name'
else:
    raise KeyError("metric.csv missing kpi_name / KPI_name column")

# Ensure numeric 'value'
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

# 1) Compute global P95 per (cmdb_id, kpi_name) using the full file (global thresholds)
global_p95 = (
    metric_df
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)['value']
    .agg(global_p95=lambda x: x.quantile(0.95))
)

# 2) Filter metric.csv to incident window and target cmdb_ids
start_window = tz.localize(pd.Timestamp("2021-03-24 19:30:00"))
end_window = tz.localize(pd.Timestamp("2021-03-24 20:00:00"))

targets = ['Mysql01', 'Mysql02']

window_df = metric_df[
    (metric_df['timestamp'] >= start_window) &
    (metric_df['timestamp'] < end_window) &
    (metric_df['cmdb_id'].isin(targets))
].copy()

# 3) Merge global_p95 into window rows
window_with_p95 = window_df.merge(global_p95, on=['cmdb_id', 'kpi_name'], how='left')

# 4) Define anomaly condition per instructions:
# anomaly if (global_p95 > 0 and value >= global_p95) OR (global_p95 == 0 and value > 0)
gp = window_with_p95['global_p95']
val = window_with_p95['value']
cond = ((gp > 0) & (val >= gp)) | ((gp == 0) & (val > 0))
anomaly_rows = window_with_p95[cond].copy()

# 5) Aggregate per (cmdb_id, kpi_name)
if not anomaly_rows.empty:
    agg = anomaly_rows.groupby(['cmdb_id', 'kpi_name'], as_index=False).agg(
        count_anomalies=('value', 'count'),
        earliest_anomaly_timestamp=('timestamp', 'min'),
        max_value_in_window=('value', 'max'),
        global_p95=('global_p95', 'first')
    )
    # severity_ratio with handling for global_p95 == 0
    def compute_sev(row):
        gpv = row['global_p95']
        mv = row['max_value_in_window']
        if pd.isna(gpv):
            return np.nan
        if gpv == 0:
            return np.inf if mv > 0 else np.nan
        return mv / gpv
    agg['severity_ratio'] = agg.apply(compute_sev, axis=1)
else:
    agg = pd.DataFrame(columns=['cmdb_id','kpi_name','count_anomalies','earliest_anomaly_timestamp','max_value_in_window','global_p95','severity_ratio'])

# 6) Determine per-instance summaries
# total distinct kpis scanned for this instance (based on presence in the window)
distinct_kpis = window_df.groupby('cmdb_id')['kpi_name'].nunique().reindex(targets, fill_value=0).reset_index()
distinct_kpis.columns = ['cmdb_id', 'total_distinct_kpis_scanned']

# KPIs with count_anomalies == 30
count30 = agg[agg['count_anomalies'] == 30].copy()

rows = []
for cmdb in targets:
    total_distinct = int(distinct_kpis.loc[distinct_kpis['cmdb_id'] == cmdb, 'total_distinct_kpis_scanned'].iloc[0])
    subset = count30[count30['cmdb_id'] == cmdb]
    num_kpis_30 = int(subset.shape[0])
    if num_kpis_30 > 0:
        # earliest anomaly timestamp among count30
        earliest_ts = subset['earliest_anomaly_timestamp'].min()
        # up to 20 example kpi names
        examples = subset['kpi_name'].drop_duplicates().head(20).tolist()
        examples_str = '; '.join(examples)
    else:
        earliest_ts = pd.NaT
        examples_str = ''
    rows.append({
        'cmdb_id': cmdb,
        'total_distinct_kpis_scanned': total_distinct,
        'num_kpis_with_count30': num_kpis_30,
        'earliest_anomaly_timestamp_among_count30': earliest_ts,
        'example_kpis_count30_up_to_20': examples_str
    })

per_instance_df = pd.DataFrame(rows)

# 7) Prepare detailed table for KPI groups that have count_anomalies == 30
# select and order columns as requested; sort by cmdb_id and limit to top 50
if not count30.empty:
    count30_details = count30[['cmdb_id','kpi_name','global_p95','count_anomalies','earliest_anomaly_timestamp','max_value_in_window','severity_ratio']].copy()
    count30_details = count30_details.sort_values(by=['cmdb_id','kpi_name']).reset_index(drop=True).head(50)
else:
    count30_details = pd.DataFrame(columns=['cmdb_id','kpi_name','global_p95','count_anomalies','earliest_anomaly_timestamp','max_value_in_window','severity_ratio'])

# Keep outputs compact per rules: per_instance_df (max 2 rows) and count30_details (up to 50 rows)
per_instance_df, count30_details.head(50)

```
Out[9]:
```
```python
# Compose a focused plain-English summary using previously computed results:
# Reuse variables from the kernel: per_instance_df, count30_details

per = per_instance_df.copy()
details = count30_details.copy()

# Ensure expected variables exist
if 'per' not in globals() or per.empty:
    summary = "No results available for Mysql01/Mysql02."
else:
    # Extract per-instance info
    lines = []
    for row in per.itertuples(index=False):
        cmdb = row.cmdb_id
        total_kpis = int(row.total_distinct_kpis_scanned)
        num_count30 = int(row.num_kpis_with_count30)
        earliest = row.earliest_anomaly_timestamp_among_count30
        earliest_str = earliest.strftime('%Y-03-24 %H:%M:%S%z') if pd.notna(earliest) else "None"
        examples = row.example_kpis_count30_up_to_20
        examples_display = examples if examples else "None"
        lines.append(
            f"{cmdb}: scanned {total_kpis} distinct KPIs; "
            f"{num_count30} KPI(s) were anomalous for every minute of the 30-minute window; "
            f"earliest anomaly among these = {earliest_str}; "
            f"example KPIs (up to 20) = {examples_display}"
        )
    # Summary of detailed list size
    details_count = int(details.shape[0]) if 'details' in globals() else 0
    lines.append(f"Total KPI groups with count_anomalies == 30 across both instances: {details_count}.")
    summary = "Summary:\n- " + "\n- ".join(lines)

# Display the concise summary string
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(   cmdb_id  total_distinct_kpis_scanned  num_kpis_with_count30 earliest_anomaly_timestamp_among_count30                      example_kpis_count30_up_to_20
0  Mysql01                          223                     17                2021-03-24 19:30:00+08:00  Mysql-MySQL_3306_Binlog stmt cache use; Mysql-...
1  Mysql02                          223                     13                2021-03-24 19:30:00+08:00  Mysql-MySQL_3306_Binlog stmt cache use; Mysql-...,     cmdb_id                                        kpi_name  global_p95  count_anomalies earliest_anomaly_timestamp  max_value_in_window  severity_ratio
0   Mysql01          Mysql-MySQL_3306_Binlog stmt cache use       360.0               30  2021-03-24 19:30:00+08:00                360.0             1.0
1   Mysql01      Mysql-MySQL_3306_GetConnectedStateOfMysqld         1.0               30  2021-03-24 19:30:00+08:00                  1.0             1.0
2   Mysql01  Mysql-MySQL_3306_Innodb buffer pool pages free    252197.0               30  2021-03-24 19:30:00+08:00             252197.0             1.0
3   Mysql01         Mysql-MySQL_3306_Innodb open files  num       165.0               30  2021-03-24 19:30:00+08:00                165.0             1.0
4   Mysql01       Mysql-MySQL_3306_Innodb row lock time avg       336.0               30  2021-03-24 19:30:00+08:00                336.0             1.0
..      ...                                             ...         ...              ...                        ...                  ...             ...
25  Mysql02             Mysql-MySQL_3306_Qcache Free Blocks         1.0               30  2021-03-24 19:30:00+08:00                  1.0             1.0
26  Mysql02             Mysql-MySQL_3306_Qcache Free Memory   1031832.0               30  2021-03-24 19:30:00+08:00            1031832.0             1.0
27  Mysql02            Mysql-MySQL_3306_Qcache Total Blocks         1.0               30  2021-03-24 19:30:00+08:00                  1.0             1.0
28  Mysql02                 Mysql-MySQL_3306_Threads Cached        12.0               30  2021-03-24 19:30:00+08:00                 12.0             1.0
29  Mysql02               Mysql-MySQL_3306_ThreadsConnected        25.0               30  2021-03-24 19:30:00+08:00                 25.0             1.0

[30 rows x 7 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[10]:

import pandas as pd
import numpy as np
import pytz
import re

tz = pytz.timezone('Asia/Shanghai')

# Reuse metric_df if present; otherwise load
if 'metric_df' not in globals():
    metric_df = pd.read_csv("metric.csv")

# Ensure timestamp is timezone-aware in UTC+8
if 'timestamp' in metric_df.columns:
    if pd.api.types.is_integer_dtype(metric_df['timestamp']) or pd.api.types.is_float_dtype(metric_df['timestamp']):
        metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    else:
        metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], utc=True, errors='coerce')
        if metric_df['timestamp'].dt.tz is None:
            metric_df['timestamp'] = metric_df['timestamp'].dt.tz_localize('UTC').dt.tz_convert(tz)
        else:
            metric_df['timestamp'] = metric_df['timestamp'].dt.tz_convert(tz)

# Standardize KPI column name to 'kpi_name'
if 'kpi_name' not in metric_df.columns and 'KPI_name' in metric_df.columns:
    metric_df = metric_df.rename(columns={'KPI_name': 'kpi_name'})
if 'kpi_name' not in metric_df.columns:
    raise KeyError("metric.csv missing kpi_name / KPI_name column")

# Ensure numeric 'value'
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

# 1) Compute global P95 per (cmdb_id, kpi_name) using the full file (do NOT filter by time)
if 'global_p95' not in globals():
    global_p95 = (
        metric_df
        .groupby(['cmdb_id', 'kpi_name'], as_index=False)['value']
        .agg(global_p95=lambda x: x.quantile(0.95))
    )

# 2) Filter to incident window and target cmdb_ids
start_window = tz.localize(pd.Timestamp("2021-03-24 19:30:00"))
end_window = tz.localize(pd.Timestamp("2021-03-24 20:00:00"))
targets = ['Mysql01', 'Mysql02']

window_df = metric_df[
    (metric_df['timestamp'] >= start_window) &
    (metric_df['timestamp'] < end_window) &
    (metric_df['cmdb_id'].isin(targets))
].copy()

# 3) Filter KPIs by case-insensitive substring matches
patterns = [
    'Innodb data pending writes',
    'Innodb data pending fsyncs',
    'CurrentSQLMaxRunningTime'
]
# Build safe regex (escape each pattern)
pattern_regex = "(" + ")|(".join(re.escape(p) for p in patterns) + ")"

kpi_mask = window_df['kpi_name'].astype(str).str.contains(pattern_regex, case=False, regex=True, na=False)
matched_window = window_df[kpi_mask].copy()

# Output A: time series rows inside the window with required columns, sorted and limited to 200 rows
timeseries_cols = ['timestamp', 'cmdb_id', 'kpi_name', 'value']
timeseries_df = matched_window.loc[:, timeseries_cols].sort_values(by=['cmdb_id', 'timestamp'], ascending=[True, True]).head(200).reset_index(drop=True)

# 4) Prepare aggregates per (cmdb_id, kpi_name)
# Merge global_p95 (precomputed) to ensure global thresholds are used
merged = matched_window.merge(global_p95, on=['cmdb_id', 'kpi_name'], how='left')

# For each group compute:
# - count_nonzero_in_window: number of rows with value > 0
# - earliest_nonzero_timestamp_in_window: earliest timestamp where value > 0
# - max_value_in_window
# - severity_ratio: max_value_in_window / global_p95 (inf if global_p95==0 and max>0)
def compute_group_aggregates(df):
    # Count non-zero
    nonzero = df[df['value'] > 0]
    count_nonzero = int(nonzero.shape[0])
    if count_nonzero > 0:
        earliest_nonzero = nonzero['timestamp'].min()
        max_val = nonzero['value'].max()
    else:
        earliest_nonzero = pd.NaT
        # even if all zero, max_value_in_window should reflect max (could be 0 or NaN)
        max_val = df['value'].max() if not df['value'].isna().all() else np.nan

    gp = df['global_p95'].iloc[0] if 'global_p95' in df.columns else np.nan
    # severity ratio handling
    if pd.isna(gp):
        sev = np.nan
    else:
        if gp == 0:
            sev = np.inf if (not pd.isna(max_val) and max_val > 0) else (np.nan if (pd.isna(max_val) or max_val == 0) else np.nan)
        else:
            sev = max_val / gp if not pd.isna(max_val) else np.nan

    return pd.Series({
        'global_p95': gp,
        'count_nonzero_in_window': count_nonzero,
        'earliest_nonzero_timestamp_in_window': earliest_nonzero,
        'max_value_in_window': max_val,
        'severity_ratio': sev
    })

if not merged.empty:
    agg = merged.groupby(['cmdb_id', 'kpi_name']).apply(compute_group_aggregates).reset_index()
    # Ensure timestamps are tz-aware in Asia/Shanghai
    if 'earliest_nonzero_timestamp_in_window' in agg.columns:
        # They should already be tz-aware, but ensure dtype
        agg['earliest_nonzero_timestamp_in_window'] = pd.to_datetime(agg['earliest_nonzero_timestamp_in_window']).dt.tz_convert(tz)
    # Limit to top 20 groups (sort by cmdb_id then kpi_name)
    agg_summary = agg.sort_values(by=['cmdb_id', 'kpi_name']).head(20).reset_index(drop=True)
else:
    agg_summary = pd.DataFrame(columns=['cmdb_id','kpi_name','global_p95','count_nonzero_in_window','earliest_nonzero_timestamp_in_window','max_value_in_window','severity_ratio'])

# Return compact results: timeseries (A) and aggregated summary per group (B)
timeseries_df, agg_summary

```
Out[10]:
```
```python
# Summarize the filtered timeseries and aggregates for the three requested KPI patterns on Mysql01/Mysql02.
# Reuse existing variables from the kernel: timeseries_df, agg_summary

import numpy as np
import pandas as pd

# Build summary lines
total_rows = int(timeseries_df.shape[0]) if 'timeseries_df' in globals() else 0
per_cmdb_counts = timeseries_df.groupby('cmdb_id').size().to_dict() if total_rows>0 else {}

lines = []
lines.append("Summary of requested KPIs in metric.csv for 2021-03-24 19:30–20:00 (UTC+8), cmdb_id in {Mysql01, Mysql02}:")
lines.append(f"- Total time-series rows returned (limited to 200): {total_rows}.")
for cmdb, cnt in per_cmdb_counts.items():
    lines.append(f"  - {cmdb}: {cnt} rows in the window.")

lines.append("- Per-(cmdb_id, kpi_name) aggregate highlights (global_p95, count_nonzero_in_window, earliest_nonzero_timestamp, max_value_in_window, severity_ratio):")

if 'agg_summary' in globals() and not agg_summary.empty:
    for _, r in agg_summary.iterrows():
        cmdb = r['cmdb_id']
        kpi = r['kpi_name']
        gp = r['global_p95']
        cnt_nz = int(r['count_nonzero_in_window']) if not pd.isna(r['count_nonzero_in_window']) else 0
        ts = r['earliest_nonzero_timestamp_in_window']
        ts_str = pd.NaT if (pd.isna(ts) or str(ts) == 'NaT') else pd.Timestamp(ts).strftime('%Y-%m-%d %H:%M:%S%z')
        maxv = r['max_value_in_window']
        sev = r['severity_ratio']
        if pd.isna(sev):
            sev_str = "NaN"
        elif np.isinf(sev):
            sev_str = "inf"
        else:
            sev_str = f"{float(sev):.3g}"
        lines.append(f"  - {cmdb} / {kpi}: global_p95={gp}, count_nonzero={cnt_nz}, earliest_nonzero={ts_str}, max={maxv}, severity_ratio={sev_str}")
else:
    lines.append("  - No matching KPI groups found in the window.")

# Compact conclusion / notable points
lines.append("Notable observations:")
# look for the specific notable rows programmatically
if 'agg_summary' in globals() and not agg_summary.empty:
    # Find Mysql02 Innodb data pending fsyncs
    cond = (agg_summary['cmdb_id']=='Mysql02') & (agg_summary['kpi_name'].str.contains('Innodb data pending fsyncs', case=False))
    if cond.any():
        row = agg_summary[cond].iloc[0]
        lines.append(f"  - Mysql02 'Innodb data pending fsyncs' had {int(row['count_nonzero_in_window'])} non-zero points (earliest {pd.Timestamp(row['earliest_nonzero_timestamp_in_window']).strftime('%Y-%m-%d %H:%M:%S%z')}), max={row['max_value_in_window']}, severity_ratio={('inf' if np.isinf(row['severity_ratio']) else f'{float(row['severity_ratio']):.3g}')}.")
    # Mysql01 had a single non-zero for the same metric
    cond1 = (agg_summary['cmdb_id']=='Mysql01') & (agg_summary['kpi_name'].str.contains('Innodb data pending fsyncs', case=False))
    if cond1.any():
        row = agg_summary[cond1].iloc[0]
        lines.append(f"  - Mysql01 'Innodb data pending fsyncs' had {int(row['count_nonzero_in_window'])} non-zero point (earliest {('None' if pd.isna(row['earliest_nonzero_timestamp_in_window']) else pd.Timestamp(row['earliest_nonzero_timestamp_in_window']).strftime('%Y-%m-%d %H:%M:%S%z'))}), max={row['max_value_in_window']}, severity_ratio={('inf' if np.isinf(row['severity_ratio']) else f'{float(row['severity_ratio']):.3g}')}.")

lines.append("End of summary.")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                    timestamp  cmdb_id                                     kpi_name  value
0   2021-03-24 19:30:00+08:00  Mysql01    Mysql-MySQL_3306_CurrentSQLMaxRunningTime    0.0
1   2021-03-24 19:30:00+08:00  Mysql01  Mysql-MySQL_3306_Innodb data pending fsyncs    0.0
2   2021-03-24 19:30:00+08:00  Mysql01  Mysql-MySQL_3306_Innodb data pending writes    0.0
3   2021-03-24 19:31:00+08:00  Mysql01    Mysql-MySQL_3306_CurrentSQLMaxRunningTime    0.0
4   2021-03-24 19:31:00+08:00  Mysql01  Mysql-MySQL_3306_Innodb data pending fsyncs    0.0
..                        ...      ...                                          ...    ...
175 2021-03-24 19:58:00+08:00  Mysql02  Mysql-MySQL_3306_Innodb data pending fsyncs    1.0
176 2021-03-24 19:58:00+08:00  Mysql02  Mysql-MySQL_3306_Innodb data pending writes    0.0
177 2021-03-24 19:59:00+08:00  Mysql02    Mysql-MySQL_3306_CurrentSQLMaxRunningTime    0.0
178 2021-03-24 19:59:00+08:00  Mysql02  Mysql-MySQL_3306_Innodb data pending fsyncs    1.0
179 2021-03-24 19:59:00+08:00  Mysql02  Mysql-MySQL_3306_Innodb data pending writes    0.0

[180 rows x 4 columns],    cmdb_id                                     kpi_name  global_p95  count_nonzero_in_window earliest_nonzero_timestamp_in_window  max_value_in_window  severity_ratio
0  Mysql01    Mysql-MySQL_3306_CurrentSQLMaxRunningTime         0.0                        0                                  NaT                  0.0             NaN
1  Mysql01  Mysql-MySQL_3306_Innodb data pending fsyncs         0.0                        1            2021-03-24 19:50:00+08:00                  1.0             inf
2  Mysql01  Mysql-MySQL_3306_Innodb data pending writes         0.0                        0                                  NaT                  0.0             NaN
3  Mysql02    Mysql-MySQL_3306_CurrentSQLMaxRunningTime         0.0                        1            2021-03-24 19:52:00+08:00                  2.0             inf
4  Mysql02  Mysql-MySQL_3306_Innodb data pending fsyncs         1.0                       17            2021-03-24 19:30:00+08:00                  2.0             2.0
5  Mysql02  Mysql-MySQL_3306_Innodb data pending writes         0.0                        1            2021-03-24 19:45:00+08:00                  1.0             inf)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```